# Project 2: Moderate — Causal Mini-GPT (Autoregressive Pattern & Code Engine)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

In [2]:
# Seed
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
def generate_math_data(num_samples=4000):
    lines = []
    ops = {'+': lambda a, b: a + b, '-': lambda a, b: a - b, '*': lambda a, b: a * b}
    
    for _ in range(num_samples):
        op = np.random.choice(['+', '-', '*'])
        if op == '*':
            # Keep numbers single digit for multiplication to stay in single-line integer range
            a = np.random.randint(0, 10)
            b = np.random.randint(0, 10)
        elif op == '-':
            a = np.random.randint(0, 20)
            b = np.random.randint(0, a + 1)  # keep result non-negative
        else:
            a = np.random.randint(0, 50)
            b = np.random.randint(0, 50)
            
        res = ops[op](a, b)
        # Format: eval(a+b)=res;
        lines.append(f"eval({a}{op}{b})={res};")
    return " ".join(lines)

raw_text = generate_math_data(num_samples=3000)
print("Preview of training text:\n", raw_text[:120])

Preview of training text:
 eval(3*7)=21; eval(20+38)=58; eval(18-10)=8; eval(7*4)=28; eval(5*4)=20; eval(11-5)=6; eval(0-0)=0; eval(11-8)=3; eval(2


In [4]:
# Build vocabulary
vocab = sorted(list(set(raw_text)))
vocab_size = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}

print(f"Vocab Size: {vocab_size} characters -> {vocab}")

# Encode entire string to integer indices
encoded_text = np.array([char_to_idx[c] for c in raw_text], dtype=np.int32)

# Context window length
seq_len =32

# Create training sequences with a stride of 3 for fast coverage
step = 3
input_list, target_list = [],[]

for i in range(0, len(encoded_text)-seq_len-1, step):
    input_list.append(encoded_text[i: i+seq_len])
    # The target is the EXACT same sequence shifted right by 1 token
    target_list.append(encoded_text[i+1: i+seq_len+1])

x_train = np.array(input_list)
y_train =  np.array(target_list)

print(f"X_train shape: {x_train.shape}, y_train shape: {y_train.shape}")

Vocab Size: 22 characters -> [' ', '(', ')', '*', '+', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ';', '=', 'a', 'e', 'l', 'v']
X_train shape: (14333, 32), y_train shape: (14333, 32)


In [5]:
class CausalTransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model // num_heads
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="gelu"),
            tf.keras.layers.Dense(d_model)
        ])
        
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # 1. Pre-LN + Causal Self-Attention
        norm1 = self.ln1(inputs)
        attn_out = self.mha(
            query=norm1, 
            value=norm1, 
            key=norm1, 
            use_causal_mask=True
        )
        x = inputs + self.dropout1(attn_out, training=training)
        
        # 2. Pre-LN + Feed-Forward
        norm2 = self.ln2(x)
        ffn_out = self.ffn(norm2)
        x = x + self.dropout2(ffn_out, training=training)
        return x

In [6]:
class GPTEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(maxlen, embed_dim)
        self.maxlen = maxlen

    def call(self,x):
        length = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=length, delta=1)

        return self.token_emb(x) + self.pos_emb(positions)

In [7]:
embed_dim = 64
num_heads = 4
d_ff = 128

inputs = tf.keras.layers.Input(shape=(seq_len,), dtype=tf.int32)
x = GPTEmbedding(maxlen=seq_len, vocab_size=vocab_size, embed_dim=embed_dim)(inputs)

# Stack 2 Causal Transformer Blocks
x = CausalTransformerBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)(x)
x = CausalTransformerBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)(x)

x = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x)
outputs = tf.keras.layers.Dense(vocab_size, activation="softmax")(x)

gpt_model = tf.keras.Model(inputs=inputs, outputs=outputs)

gpt_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gpt_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gpt_embedding (GPTEmbedding)         │ (None, 32, 64)              │           3,456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ causal_transformer_block             │ (None, 32, 64)              │          33,472 │
│ (CausalTransformerBlock)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ causal_transformer_block_1           │ (None, 32, 64)              │          33,472 │
│ (CausalTransformerBlock)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization_4                │ (None, 32, 64)              │             128 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32, 22)              │           1,430 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 71,958 (281.09 KB)

 Trainable params: 71,958 (281.09 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
hisotry = gpt_model.fit(
    x_train,y_train,
    batch_size=64,
    epochs=15,
    verbose=1
    
)

Epoch 1/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 16s 38ms/step - accuracy: 0.6800 - loss: 0.9841
Epoch 2/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.7311 - loss: 0.7566
Epoch 3/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.7431 - loss: 0.7177
Epoch 4/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - accuracy: 0.7549 - loss: 0.6843
Epoch 5/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.7649 - loss: 0.6565
Epoch 6/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.7723 - loss: 0.6369
Epoch 7/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.7767 - loss: 0.6254
Epoch 8/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.7811 - loss: 0.6150
Epoch 9/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.7862 - loss: 0.6017
Epoch 10/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.7918 - loss: 0.5863
Epoch 11/15
224/224 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.7966 - loss: 0.5731
Epoch 12/15
224/224 ━━━━━━━━━━━━━━━━━━━━

In [9]:
def generate_math_text(prompt, max_new_tokens=40, temperature=0.2):
    # Convert characters to indices
    generated_indices = [char_to_idx[c] for c in prompt]
    
    for _ in range(max_new_tokens):
        # Always slice the most recent context window (seq_len = 32)
        context = generated_indices[-seq_len:]
        
        # Left-pad if the prompt is shorter than seq_len
        if len(context) < seq_len:
            padded_context = [0] * (seq_len - len(context)) + context
        else:
            padded_context = context
            
        input_tensor = np.array([padded_context], dtype=np.int32)
        
        # Predict: shape (1, seq_len, vocab_size)
        preds = gpt_model.predict(input_tensor, verbose=0)
        
        # Extract logits for the very last timestep
        next_token_logits = preds[0, -1, :]
        
        # Temperature sampling
        if temperature == 0.0:
            next_idx = int(np.argmax(next_token_logits))
        else:
            # Scaled log-probabilities
            logits = np.log(next_token_logits + 1e-9) / temperature
            probs = np.exp(logits) / np.sum(np.exp(logits))
            next_idx = int(np.random.choice(len(probs), p=probs))
            
        generated_indices.append(next_idx)
        
        # Stop early if a semi-colon is reached after completing an evaluation
        if idx_to_char[next_idx] == ';' and len(generated_indices) > len(prompt) + 2:
            break
            
    return "".join([idx_to_char[i] for i in generated_indices])

In [10]:
test_prompts = [
    "eval(7*8)=",
    "eval(12+15)=",
    "eval(18-6)=",
    "eval(4*5)="
]

for p in test_prompts:
    completed = generate_math_text(p, max_new_tokens=10, temperature=0.0)
    print(f"Prompt: '{p}'  -->  Generated: '{completed}'")

Prompt: 'eval(7*8)='  -->  Generated: 'eval(7*8)=56;'
Prompt: 'eval(12+15)='  -->  Generated: 'eval(12+15)=27;'
Prompt: 'eval(18-6)='  -->  Generated: 'eval(18-6)=12;'
Prompt: 'eval(4*5)='  -->  Generated: 'eval(4*5)=20;'
